# Multi-Provider 설정: 하나의 Manager에서 Coinbase + Stripe(Privy) 사용

## 개요

Tutorial 00에서는 단일 wallet provider를 설정합니다. 이 Notebook에서는 하나의 Payment Manager에 Coinbase CDP와 Stripe(Privy) connector를 모두 연결하는 **multi-connector pattern**을 보여줍니다. 서로 다른 사용자 또는 동일한 사용자가 각기 다른 provider의 wallet을 가질 수 있으며, 모두 같은 payment stack을 통해 관리됩니다.



```
Payment Manager(공유)
  ├── Coinbase CDP Connector
  │     └── Embedded Wallet(사용자 A)
  ├── StripePrivy Connector
  │     └── Embedded Wallet(사용자 B)
  └── Payment Session(예산 — 어느 wallet에서든 사용 가능)
```

### 사전 요구 사항

* `.env`에 Coinbase CDP 및 Privy credentials 모두 설정
* IAM role: Tutorial 00에서 자동 생성(`setup_payment_roles()`)

> **Testnet 전용입니다.** 모든 코드는 [faucet.circle.com](https://faucet.circle.com/)에서 무료 USDC를 받아 Base Sepolia 또는 Solana Devnet을 사용합니다. Testnet USDC는 현실 세계의 가치가 없습니다.


In [ ]:
%pip install -r requirements.txt --quiet

In [ ]:
import os
import uuid
import sys

sys.path.append("..")
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

import boto3
from utils import (
    assume_role,
    idempotent_create,
    wait_for_status,
    client_token,
    save_tutorial_config,
    print_summary,
    require_env,
)

AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")
# os.environ['AWS_PROFILE'] = '<your-profile>'

CP_ENDPOINT = os.environ.get(
    "PAYMENTS_CP_ENDPOINT",
    f"https://bedrock-agentcore-control.{AWS_REGION}.amazonaws.com",
)
DP_ENDPOINT = os.environ.get("PAYMENTS_DP_ENDPOINT", f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com")
CRED_ENDPOINT = os.environ.get("CREDENTIAL_PROVIDER_ENDPOINT", CP_ENDPOINT)
NETWORK = os.environ.get("NETWORK", "ETHEREUM")

CP_ROLE_ARN = os.environ["CONTROL_PLANE_ROLE_ARN"]
MGMT_ROLE_ARN = os.environ["MANAGEMENT_ROLE_ARN"]
RR_ROLE_ARN = os.environ["RESOURCE_RETRIEVAL_ROLE_ARN"]
USER_ID = os.environ.get("USER_ID", "test-user-001")
LINKED_EMAIL = os.environ.get("LINKED_EMAIL", "")

session = boto3.Session(region_name=AWS_REGION)


# --- 사전 요구 값 출력 ---
def _check(label, value, redact=False):
    ok = bool(value) and not value.startswith("<")
    icon = "✅" if ok else "❌ MISSING"
    display = "[redacted]" if redact and value else value
    print(f"  {icon}  {label}: {display}")


print("Prerequisites (from .env / Tutorial 00):\n")
print("  AWS:")
_check("AWS_REGION", AWS_REGION)
_check("CP_ENDPOINT", CP_ENDPOINT)
_check("DP_ENDPOINT", DP_ENDPOINT)

print("\n  IAM Roles:")
_check("CONTROL_PLANE_ROLE_ARN", CP_ROLE_ARN)
_check("MANAGEMENT_ROLE_ARN", MGMT_ROLE_ARN)
_check("RESOURCE_RETRIEVAL_ROLE_ARN", RR_ROLE_ARN)

print("\n  Identity:")
_check("USER_ID", USER_ID)
_check("LINKED_EMAIL", LINKED_EMAIL)
_check("NETWORK", NETWORK)

print("\n  Coinbase CDP Credentials:")
_check("COINBASE_API_KEY_ID", os.environ.get("COINBASE_API_KEY_ID", ""))
_check(
    "COINBASE_API_KEY_SECRET",
    os.environ.get("COINBASE_API_KEY_SECRET", ""),
    redact=True,
)
_check("COINBASE_WALLET_SECRET", os.environ.get("COINBASE_WALLET_SECRET", ""), redact=True)

print("\n  Stripe (Privy) Credentials:")
_check("PRIVY_APP_ID", os.environ.get("PRIVY_APP_ID", ""))
_check("PRIVY_APP_SECRET", os.environ.get("PRIVY_APP_SECRET", ""), redact=True)
_check("PRIVY_AUTHORIZATION_ID", os.environ.get("PRIVY_AUTHORIZATION_ID", ""))
_check(
    "PRIVY_AUTHORIZATION_PRIVATE_KEY",
    os.environ.get("PRIVY_AUTHORIZATION_PRIVATE_KEY", ""),
    redact=True,
)

## 1단계 — 공유 Payment Manager 하나 생성

> **비용 안내:** Payment Manager, Connector, Instrument가 provision된 동안 AWS 요금이 발생합니다. 작업을 마치면 리소스를 정리하세요.

In [ ]:
cp_session = assume_role(session, CP_ROLE_ARN, "multi-provider-cp")
cp_client = cp_session.client("bedrock-agentcore-control", endpoint_url=CP_ENDPOINT)
cred_client = cp_session.client("bedrock-agentcore-control", endpoint_url=CRED_ENDPOINT)

suffix = uuid.uuid4().hex[:8]
MANAGER_NAME = f"MultiProviderMgr{suffix}"

resp = idempotent_create(
    cp_client.create_payment_manager,
    f"Manager '{MANAGER_NAME}' already exists",
    name=MANAGER_NAME,
    authorizerType="AWS_IAM",
    roleArn=RR_ROLE_ARN,
    clientToken=client_token(),
)
MANAGER_ID = resp["paymentManagerId"]
MANAGER_ARN = resp["paymentManagerArn"]
print(f"✅ Manager: {MANAGER_ID}")

wait_for_status(cp_client.get_payment_manager, "READY", paymentManagerId=MANAGER_ID)

## 2단계 — Coinbase CDP Connector 연결

In [ ]:
# Coinbase 자격 증명 공급자
cb_cred = cred_client.create_payment_credential_provider(
    name=f"CoinbaseCdp{suffix}",
    credentialProviderVendor="CoinbaseCDP",
    providerConfigurationInput={
        "coinbaseCdpConfiguration": {
            "apiKeyId": require_env("COINBASE_API_KEY_ID"),
            "apiKeySecret": require_env("COINBASE_API_KEY_SECRET"),
            "walletSecret": require_env("COINBASE_WALLET_SECRET"),
        }
    },
)
CB_CRED_ARN = cb_cred["credentialProviderArn"]

# Coinbase 커넥터
cb_conn = cp_client.create_payment_connector(
    paymentManagerId=MANAGER_ID,
    name=f"CoinbaseConn{suffix}",
    type="CoinbaseCDP",
    credentialProviderConfigurations=[{"coinbaseCDP": {"credentialProviderArn": CB_CRED_ARN}}],
    clientToken=client_token(),
)
CB_CONNECTOR_ID = cb_conn["paymentConnectorId"]
print(f"✅ Coinbase connector: {CB_CONNECTOR_ID}")
wait_for_status(
    cp_client.get_payment_connector,
    "READY",
    paymentManagerId=MANAGER_ID,
    paymentConnectorId=CB_CONNECTOR_ID,
)

## 3단계 — StripePrivy Connector 연결

In [ ]:
# StripePrivy 자격 증명 공급자
sp_cred = cred_client.create_payment_credential_provider(
    name=f"StripePrivy{suffix}",
    credentialProviderVendor="StripePrivy",
    providerConfigurationInput={
        "stripePrivyConfiguration": {
            "appId": require_env("PRIVY_APP_ID"),
            "appSecret": require_env("PRIVY_APP_SECRET"),
            "authorizationId": require_env("PRIVY_AUTHORIZATION_ID"),
            "authorizationPrivateKey": require_env("PRIVY_AUTHORIZATION_PRIVATE_KEY"),
        }
    },
)
SP_CRED_ARN = sp_cred["credentialProviderArn"]

# StripePrivy 커넥터
sp_conn = cp_client.create_payment_connector(
    paymentManagerId=MANAGER_ID,
    name=f"StripePrivyConn{suffix}",
    type="StripePrivy",
    credentialProviderConfigurations=[{"stripePrivy": {"credentialProviderArn": SP_CRED_ARN}}],
    clientToken=client_token(),
)
SP_CONNECTOR_ID = sp_conn["paymentConnectorId"]
print(f"✅ StripePrivy connector: {SP_CONNECTOR_ID}")
wait_for_status(
    cp_client.get_payment_connector,
    "READY",
    paymentManagerId=MANAGER_ID,
    paymentConnectorId=SP_CONNECTOR_ID,
)

## 4단계 — 두 Provider의 Instrument 생성

동일한 manager에 서로 다른 connector를 연결하면 서로 다른 wallet provider를 사용할 수 있습니다. 둘 다 `EMBEDDED_CRYPTO_WALLET`을 사용합니다.

In [ ]:
mgmt_session = assume_role(session, MGMT_ROLE_ARN, "multi-provider-mgmt")
dp_client = mgmt_session.client("bedrock-agentcore", endpoint_url=DP_ENDPOINT)

LINKED_EMAIL = os.environ.get("LINKED_EMAIL", "")
assert LINKED_EMAIL and not LINKED_EMAIL.startswith("<") and LINKED_EMAIL != "user@example.com", (
    "Set LINKED_EMAIL in .env to your real email before running this notebook."
)

instruments = {}

for label, conn_id in [
    ("coinbase", CB_CONNECTOR_ID),
    ("stripe_privy", SP_CONNECTOR_ID),
]:
    resp = dp_client.create_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=conn_id,
        userId=USER_ID,
        paymentInstrumentType="EMBEDDED_CRYPTO_WALLET",
        paymentInstrumentDetails={
            "embeddedCryptoWallet": {
                "network": NETWORK,
                "linkedAccounts": [{"email": {"emailAddress": LINKED_EMAIL}}],
            }
        },
        clientToken=client_token(),
    )
    inst = resp["paymentInstrument"]
    inst_id = inst["paymentInstrumentId"]
    wallet = inst["paymentInstrumentDetails"]["embeddedCryptoWallet"].get("walletAddress", "pending...")
    instruments[label] = {
        "instrument_id": inst_id,
        "wallet_address": wallet,
        "connector_id": conn_id,
    }
    print(f"✅ {label}: {inst_id} → {wallet}")

    wait_for_status(
        dp_client.get_payment_instrument,
        "ACTIVE",
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=conn_id,
        paymentInstrumentId=inst_id,
        userId=USER_ID,
    )

## 5단계 — 두 Wallet에 자금 입금 및 동의 완료

[faucet.circle.com](https://faucet.circle.com/)에서 두 wallet에 자금을 입금합니다.
- `ETHEREUM` → **Base Sepolia**
- `SOLANA` → **Solana Devnet**

### StripePrivy — Privy Reference Frontend 실행

최종 사용자가 에이전트에 signing 권한을 부여하기 전에는 위 StripePrivy wallet에서 결제를 받을 수 없습니다. 아직 완료하지 않았다면 [providers/stripe_privy_account_setup.ipynb](providers/stripe_privy_account_setup.ipynb)의 **3단계**를 따르세요. `github.com/privy-io/aws-agentcore-sdk`의 Privy reference frontend 사용 과정을 안내합니다.

해당 Notebook에서 Privy reference frontend를 이미 실행 중이라면 브라우저에서 다시 load하여 위에서 생성한 StripePrivy wallet을 페이지에 반영한 다음 **Connect agent**를 한 번 선택합니다. 그러면 로그인한 사용자에게 연결된 모든 Privy wallet에 AgentCore가 *additional signer*로 추가됩니다. 나중에 다시 선택해도 안전합니다.

> **CoinbaseCDP**는 wallet별 작업이 아니라 [CDP Portal](https://portal.cdp.coinbase.com/)에서 구성한 project-level delegation을 사용합니다. 자세한 내용은 Tutorial 00의 7b단계를 참조하세요.

In [ ]:
print("Fund these wallets at https://faucet.circle.com/\n")
for label, inst in instruments.items():
    print(f"  {label:15s} → {inst['wallet_address']}")
print(f"\n  Network: {NETWORK}")
print()
print("  StripePrivy: reload the Privy reference frontend you set up in the Privy provider notebook,")
print("    then choose Connect agent once to grant AgentCore signer access on all Privy wallets")
print(f"    linked to {os.environ.get('LINKED_EMAIL', 'user@example.com')}.")

In [ ]:
# StripePrivy: signer access가 실제 wallet에 적용되었는지 확인
# CoinbaseCDP에서는 건너뜀 — delegation은 wallet별이 아니라 CDP project level에서 처리됨
from utils import verify_privy_signer_on_wallet

privy_wallet = instruments["stripe_privy"]["wallet_address"]
try:
    ok = verify_privy_signer_on_wallet(
        app_id=require_env("PRIVY_APP_ID"),
        app_secret=require_env("PRIVY_APP_SECRET"),
        wallet_address_or_id=privy_wallet,
        quorum_id=require_env("PRIVY_AUTHORIZATION_ID"),
    )
except Exception as exc:
    print(f"  ⚠️  Consent check skipped: {exc}")
    print("     Choose Connect agent in the Privy reference frontend, then re-run this cell.")
else:
    if ok:
        print(f"  ✅ Signer access granted on {privy_wallet}")
    else:
        print(f"  ❌ Signer access has NOT been granted on {privy_wallet}.")
        print("     Reload the Privy reference frontend, choose Connect agent, then re-run this cell.")
        print("     ProcessPayment requires delegated signing on this instrument to succeed.")

## 6단계 — Session 생성 및 Config 저장

In [ ]:
resp = dp_client.create_payment_session(
    paymentManagerArn=MANAGER_ARN,
    userId=USER_ID,
    expiryTimeInMinutes=60,
    limits={"maxSpendAmount": {"value": "1.0", "currency": "USD"}},
    clientToken=client_token(),
)
SESSION_ID = resp["paymentSession"]["paymentSessionId"]
print(f"✅ Session: {SESSION_ID} (budget: $1.00)")

In [ ]:
# 이후 튜토리얼을 위해 resource ID를 .env에 기록(multi-provider)
save_tutorial_config(
    {
        "PAYMENT_MANAGER_ARN": MANAGER_ARN,
        "PAYMENT_MANAGER_ID": MANAGER_ID,
        "USER_ID": USER_ID,
        "SESSION_ID": SESSION_ID,
        "NETWORK": NETWORK,
        "CREDENTIAL_PROVIDER_TYPE": "MultiProvider",
        # Coinbase
        "COINBASE_INSTRUMENT_ID": instruments["coinbase"]["instrument_id"],
        "COINBASE_WALLET_ADDRESS": instruments["coinbase"]["wallet_address"],
        "COINBASE_CONNECTOR_ID": CB_CONNECTOR_ID,
        # Stripe(Privy)
        "PRIVY_INSTRUMENT_ID": instruments["stripe_privy"]["instrument_id"],
        "PRIVY_WALLET_ADDRESS": instruments["stripe_privy"]["wallet_address"],
        "PRIVY_CONNECTOR_ID": SP_CONNECTOR_ID,
    }
)

print_summary(
    "Multi-Provider Setup Complete",
    manager_arn=MANAGER_ARN,
    coinbase_instrument=instruments["coinbase"]["instrument_id"],
    stripe_privy_instrument=instruments["stripe_privy"]["instrument_id"],
    session_id=SESSION_ID,
)
print("Downstream tutorials pick a provider via env vars:")
print("  COINBASE_INSTRUMENT_ID / PRIVY_INSTRUMENT_ID")

## 이 예제에서 보여주는 내용

하나의 Payment Manager, 두 wallet provider, 동일한 session budget을 사용합니다. Tutorial 01 이후의 agent 코드는 변경되지 않으며 plugin에 전달할 `instrument_id`만 선택합니다.

## 검증

위 셀이 오류 없이 실행되었다면 두 connector가 모두 활성 상태입니다. 위에 Coinbase CDP용과 StripePrivy용 Payment Instrument가 하나씩 출력되며, 각각 `ACTIVE` 상태와 고유한 `walletAddress`가 표시됩니다.

## 리소스 정리

> **경고:** Payment Manager를 삭제하면 모든 하위 리소스(Connector, Credential Provider, Instrument)가 영구적으로 제거되며 되돌릴 수 없습니다. 리소스를 정리하기 전에 이후의 모든 튜토리얼을 완료했는지 확인하세요.

**비용 안내:** 사용한 서비스의 비용에 관한 자세한 내용은 AgentCore pricing을 참조하세요 - https://aws.amazon.com/bedrock/agentcore/pricing/

Session은 구성된 `expiryTimeInMinutes`가 지나면 자동으로 만료됩니다. 이 튜토리얼에서 생성한 모든 payment resource를 삭제하려면 Tutorial 00(`setup_agentcore_payments.ipynb`) 하단의 cleanup 셀을 실행합니다. Payment Manager를 삭제하면 모든 하위 리소스(Connector, Instrument)도 함께 삭제됩니다.

# 축하합니다!

Tutorial 01로 계속 진행하세요. 에이전트는 어느 provider의 instrument로도 작동합니다.